# Advanced Numerical: Verifying Classical Hopfield Convergence and Hebbian Recall Limits
In this advanced example, we numerically test two claims from the Week 6 derivations: asynchronous updates produce monotone energy descent, and Hebbian recall quality degrades as storage load increases. This notebook is the computational companion to [Advanced: Why Classical Hopfield Networks Converge and Why Hebbian Learning Works](CHEME-5820-L6a-Advanced-Classical-HopfieldConvergence-HebbianLearning-Proof-Spring-2026.ipynb).

> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
> * __Verify monotone energy descent during asynchronous retrieval:__ Run a single retrieval trajectory from a noisy cue and track the network energy after each accepted flip. Use the observed energy differences to confirm the convergence-theorem prediction at the trajectory level.
> * __Quantify how storage load changes retrieval reliability:__ Sweep the memory load and estimate recall success over repeated randomized trials. Use the trend to connect crosstalk intuition from the derivation to empirical behavior.
> * __Interpret simulation diagnostics in Hopfield language:__ Explain what the printed diagnostics mean in terms of accepted flips, local minima, and stable endpoints. Distinguish between convergence to a fixed point and exact recovery of the target memory.

Let's get started!
___


## Setup, Model, and Helper Functions
First, we set up a minimal Julia simulation environment for classical Hopfield dynamics. In this notebook, all package loading and helper-function definitions are centralized in `Include.jl` and `src/Proof.jl` so the code cells stay focused on experiments.

> __What this setup implements:__
> 
> * __Centralized environment setup__: The `Include.jl` file handles package loading and includes `src/Proof.jl`. This keeps setup logic in one place and makes notebook cells easier to read.
> * __Hebbian storage and energy utilities__: The helper functions in `src/Proof.jl` implement weight construction, energy evaluation, noise injection, and asynchronous retrieval. These utilities are reused across both numerical tasks.
> * __Student-readable implementations__: Each function in `src/Proof.jl` includes docstrings and comments so you can map code steps directly to the convergence and Hebbian derivations.

Let's include the shared setup file and load the helper methods.



In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # load packages + include src/Proof.jl

async_retrieve (generic function with 1 method)

___

## Task 1: Verify Monotonic Energy Descent on One Retrieval Trajectory
In this task, we initialize the network with one noisy cue and run asynchronous retrieval until no more flips are accepted or we hit a sweep limit. We then compute observed energy differences along the accepted-flip trajectory.

> __What to look for in the output:__
> 
> * __Energy monotonicity check__: The reported `largest_delta_E` should be non-positive up to numerical tolerance. If this value is positive, then at least one accepted asynchronous update increased energy, which would contradict the convergence derivation assumptions.
> * __Retrieval quality check__: Compare initial and final Hamming distance to the selected target memory. A smaller final distance indicates that descent moved the state toward the target basin, even though exact recovery is not guaranteed in every single run.

Now run one retrieval trajectory and print the key diagnostics.


In [ ]:
let
    rng = MersenneTwister(5820)

    N = 120                      # number of neurons
    K = 10                       # number of stored memories
    flip_fraction = 0.20         # fraction of bits flipped in the cue
    max_sweeps = 80

    patterns = rand(rng, (-1, 1), K, N)
    W = hebbian_weights(patterns)

    target = vec(patterns[1, :])
    noisy = corrupt_pattern(target, flip_fraction, rng)
    retrieved, energies = async_retrieve(noisy, W; max_sweeps=max_sweeps, rng=rng)

    energy_diffs = diff(energies)
    largest_delta_E = isempty(energy_diffs) ? 0.0 : maximum(energy_diffs)
    is_monotone = all(energy_diffs .<= 1e-12)
    exact_recovery = retrieved == target

    println("Monotonic energy check (single retrieval run)")
    @printf("N = %d, K = %d, alpha = %.3f\n", N, K, K / N)
    println("Energy monotone non-increasing: ", is_monotone)
    @printf("Largest observed delta E over accepted flips: %.3e\n", largest_delta_E)
    println("Initial Hamming distance to target: ", hamming_distance(noisy, target))
    println("Final Hamming distance to target:   ", hamming_distance(retrieved, target))
    println("Number of accepted flips: ", length(energies) - 1)
    println("Exact recovery of target memory: ", exact_recovery)
end

Monotonic energy check (single retrieval run)
N = 120, K = 10, alpha = 0.083
Energy monotone non-increasing: true
Largest observed delta E over accepted flips: -3.667e-01
Initial Hamming distance to target: 24
Final Hamming distance to target:   0
Number of accepted flips: 24


The first task tests the convergence theorem at the trajectory level. If energy is monotone and `largest_delta_E` is non-positive, then the observed updates are consistent with $\Delta E\le 0$ for asynchronous accepted flips.

___

## Task 2: Measure Recall Success as Load Increases
In this task, we keep cue corruption fixed and vary load $\alpha = K/N$ across repeated randomized trials. The goal is to measure how often the network converges to the exact target memory under increasing interference.

> __Experimental design:__
> 
> * __Fixed conditions__: We keep neuron count, corruption fraction, and retrieval settings fixed while changing only load. This isolates how memory interference changes recall quality.
> * __Strict success metric__: A trial is counted as success only when the retrieved final state exactly matches the sampled target memory. This metric separates exact memory recovery from merely converging to some stable attractor.

Run the load sweep below and inspect the printed success-rate table.


In [ ]:
let

    # initialize -
    N = 120
    alpha_values = [0.03, 0.06, 0.09, 0.12, 0.15, 0.18, 0.21]
    results = recall_success_rate(N, alpha_values; trials=50, flip_fraction=0.10, max_sweeps=80, seed=5822)

    println("Recall performance vs load alpha = K/N")
    println("N = $(N), noise = 10% bit flips, trials per alpha = 50")
    println("alpha    K    success_rate")

    for (alpha, K, rate) in results
        bar = repeat("#", round(Int, 30 * rate))
        @printf("%5.2f  %3d    %6.3f  %s\n", alpha, K, rate, bar)
    end
end

Recall performance vs load alpha = K/N
N = 120, noise = 10% bit flips, trials per alpha = 50
alpha    K    success_rate
 0.03    4     1.000  ##############################
 0.06    7     1.000  ##############################
 0.09   11     0.980  #############################
 0.12   14     0.960  #############################
 0.15   18     0.620  ###################
 0.18   22     0.240  #######
 0.21   25     0.160  #####


## Summary
This advanced numerical example validates the two main derivation-level claims for classical Hopfield networks using direct Julia simulations. The experiments connect proof statements to measurable trajectory and population-level behavior.

> __Key Takeaways:__
> 
> * **Convergence mechanism in practice:** In single-trajectory runs, accepted asynchronous updates keep energy non-increasing within numerical tolerance. This matches the theorem result that the Hopfield energy acts as a Lyapunov function under asynchronous updates.
> * **Convergence and exact recall are different outcomes:** A run can converge to a stable state even when that state is not the exact target memory. Hamming-distance and exact-match diagnostics make this distinction explicit in the experiment outputs.
> * **Load controls recall reliability:** As storage load increases at fixed corruption, exact-recall success generally decreases across repeated trials. This trend is consistent with the signal-crosstalk interpretation from the advanced derivation notebook.

Together, these experiments provide a clear numerical bridge from the formal derivations to practical retrieval behavior.
___
